# Aula05.Ex03 - Formas 3D Básicas para Cena

### Primeiro, vamos importar as bibliotecas necessárias.

In [1]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math
import ctypes

### Inicializando janela

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(900, 700, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfw.terminate()

glfw.make_context_current(window)

### Eventos de teclado

In [3]:
polygonal_mode = False

t_x_cilindro = 0.75
t_y_cilindro = 0.45

escala_esfera = 0.18

angulo_cone = 0.0

def key_event(window, key, scancode, action, mods):
    global polygonal_mode
    global t_x_cilindro, t_y_cilindro
    global escala_esfera
    global angulo_cone

    if key == glfw.KEY_P and action == glfw.PRESS:
        polygonal_mode = not polygonal_mode

    if action == glfw.PRESS or action == glfw.REPEAT:

        if key == glfw.KEY_LEFT:
            t_x_cilindro -= 0.03

        if key == glfw.KEY_RIGHT:
            t_x_cilindro += 0.03

        if key == glfw.KEY_UP:
            t_y_cilindro += 0.03

        if key == glfw.KEY_DOWN:
            t_y_cilindro -= 0.03

        if key == glfw.KEY_A:
            escala_esfera += 0.01

        if key == glfw.KEY_Z:
            escala_esfera -= 0.01
            if escala_esfera < 0.05:
                escala_esfera = 0.05

        if key == glfw.KEY_Q:
            angulo_cone += 0.05

        if key == glfw.KEY_E:
            angulo_cone -= 0.05

glfw.set_key_callback(window, key_event)


### Shaders

Note que agora usamos vec3, já que estamos em 3D.

In [4]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [5]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

### Requisitando slot para a GPU para nossos programas Vertex e Fragment Shaders

In [6]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

### Associando nosso código-fonte aos slots solicitados

In [7]:
# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

### Compilando o Vertex Shader

Se há algum erro em nosso programa Vertex Shader, nosso app para por aqui.

In [8]:
# Compile shaders
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

### Compilando o Fragment Shader

Se há algum erro em nosso programa Fragment Shader, nosso app para por aqui.

In [9]:
glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

### Associando os programas compilados ao programa principal

In [10]:
# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

### Linkagem do programa

In [11]:
# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

# Make program the default program
glUseProgram(program)

### Preparando dados para enviar a GPU

Agora vamos modelar algumas formas 3D básicas.

### Funções auxiliares para modelagem

In [12]:
PI = math.pi

def adiciona_triangulo(vertices, p0, p1, p2):
    vertices.append(p0)
    vertices.append(p1)
    vertices.append(p2)

def adiciona_quad(vertices, p0, p1, p2, p3):
    vertices.append(p0)
    vertices.append(p1)
    vertices.append(p2)

    vertices.append(p0)
    vertices.append(p2)
    vertices.append(p3)

### Modelando um cubo

In [13]:
def cria_cubo(tamanho=1.0):
    h = tamanho / 2.0
    vertices = []

    p000 = (-h, -h, -h)
    p001 = (-h, -h,  h)
    p010 = (-h,  h, -h)
    p011 = (-h,  h,  h)
    p100 = ( h, -h, -h)
    p101 = ( h, -h,  h)
    p110 = ( h,  h, -h)
    p111 = ( h,  h,  h)

    adiciona_quad(vertices, p001, p101, p111, p011)
    adiciona_quad(vertices, p000, p010, p110, p100)
    adiciona_quad(vertices, p000, p001, p011, p010)
    adiciona_quad(vertices, p100, p110, p111, p101)
    adiciona_quad(vertices, p010, p011, p111, p110)
    adiciona_quad(vertices, p000, p100, p101, p001)

    return vertices

### Modelando uma pirâmide de base quadrada

In [14]:
def cria_piramide(base=1.0, altura=1.2):
    hb = base / 2.0
    z_base = -altura / 2.0
    z_topo = altura / 2.0

    vertices = []

    p0 = (-hb, -hb, z_base)
    p1 = ( hb, -hb, z_base)
    p2 = ( hb,  hb, z_base)
    p3 = (-hb,  hb, z_base)
    topo = (0.0, 0.0, z_topo)

    adiciona_quad(vertices, p0, p1, p2, p3)
    adiciona_triangulo(vertices, p0, p1, topo)
    adiciona_triangulo(vertices, p1, p2, topo)
    adiciona_triangulo(vertices, p2, p3, topo)
    adiciona_triangulo(vertices, p3, p0, topo)

    return vertices

### Modelando um prisma triangular

In [15]:
def cria_prisma_triangular(largura=1.0, altura=1.0, comprimento=1.2):
    hx = largura / 2.0
    hy = altura / 2.0
    hz = comprimento / 2.0

    vertices = []

    a0 = (-hx, -hy,  hz)
    a1 = ( hx, -hy,  hz)
    a2 = (0.0,  hy,  hz)

    b0 = (-hx, -hy, -hz)
    b1 = ( hx, -hy, -hz)
    b2 = (0.0,  hy, -hz)

    adiciona_triangulo(vertices, a0, a1, a2)
    adiciona_triangulo(vertices, b0, b2, b1)

    adiciona_quad(vertices, a0, b0, b1, a1)
    adiciona_quad(vertices, a1, b1, b2, a2)
    adiciona_quad(vertices, a2, b2, b0, a0)

    return vertices

### Modelando um cilindro

In [16]:
def cria_cilindro(raio=0.5, altura=1.2, setores=24):
    vertices = []
    z0 = -altura / 2.0
    z1 = altura / 2.0
    passo = (2.0 * PI) / setores

    centro_base = (0.0, 0.0, z0)
    centro_topo = (0.0, 0.0, z1)

    for i in range(setores):
        a0 = i * passo
        a1 = (i + 1) * passo

        p0 = (raio * math.cos(a0), raio * math.sin(a0), z0)
        p1 = (raio * math.cos(a1), raio * math.sin(a1), z0)
        p2 = (raio * math.cos(a0), raio * math.sin(a0), z1)
        p3 = (raio * math.cos(a1), raio * math.sin(a1), z1)

        adiciona_triangulo(vertices, centro_base, p1, p0)
        adiciona_triangulo(vertices, centro_topo, p2, p3)

        vertices.append(p0)
        vertices.append(p1)
        vertices.append(p3)

        vertices.append(p0)
        vertices.append(p3)
        vertices.append(p2)

    return vertices

### Modelando um cone

In [17]:
def cria_cone(raio=0.5, altura=1.2, setores=24):
    vertices = []
    z_base = -altura / 2.0
    z_topo = altura / 2.0
    passo = (2.0 * PI) / setores

    centro_base = (0.0, 0.0, z_base)
    topo = (0.0, 0.0, z_topo)

    for i in range(setores):
        a0 = i * passo
        a1 = (i + 1) * passo

        p0 = (raio * math.cos(a0), raio * math.sin(a0), z_base)
        p1 = (raio * math.cos(a1), raio * math.sin(a1), z_base)

        adiciona_triangulo(vertices, centro_base, p1, p0)
        adiciona_triangulo(vertices, p0, p1, topo)

    return vertices

### Modelando uma esfera

In [18]:
def cria_esfera(raio=0.5, setores=24, stacks=16):
    vertices = []

    for i in range(stacks):
        phi0 = -PI / 2.0 + i * PI / stacks
        phi1 = -PI / 2.0 + (i + 1) * PI / stacks

        for j in range(setores):
            theta0 = j * 2.0 * PI / setores
            theta1 = (j + 1) * 2.0 * PI / setores

            p0 = (raio * math.cos(phi0) * math.cos(theta0),
                  raio * math.cos(phi0) * math.sin(theta0),
                  raio * math.sin(phi0))

            p1 = (raio * math.cos(phi0) * math.cos(theta1),
                  raio * math.cos(phi0) * math.sin(theta1),
                  raio * math.sin(phi0))

            p2 = (raio * math.cos(phi1) * math.cos(theta1),
                  raio * math.cos(phi1) * math.sin(theta1),
                  raio * math.sin(phi1))

            p3 = (raio * math.cos(phi1) * math.cos(theta0),
                  raio * math.cos(phi1) * math.sin(theta0),
                  raio * math.sin(phi1))

            adiciona_triangulo(vertices, p0, p1, p2)
            adiciona_triangulo(vertices, p0, p2, p3)

    return vertices

### Criando as formas

In [19]:
vertices_list = []

cubo = cria_cubo()
inicio_cubo = len(vertices_list)
vertices_list.extend(cubo)
quantos_vertices_cubo = len(cubo)

piramide = cria_piramide()
inicio_piramide = len(vertices_list)
vertices_list.extend(piramide)
quantos_vertices_piramide = len(piramide)

prisma = cria_prisma_triangular()
inicio_prisma = len(vertices_list)
vertices_list.extend(prisma)
quantos_vertices_prisma = len(prisma)

cilindro = cria_cilindro()
inicio_cilindro = len(vertices_list)
vertices_list.extend(cilindro)
quantos_vertices_cilindro = len(cilindro)

cone = cria_cone()
inicio_cone = len(vertices_list)
vertices_list.extend(cone)
quantos_vertices_cone = len(cone)

esfera = cria_esfera()
inicio_esfera = len(vertices_list)
vertices_list.extend(esfera)
quantos_vertices_esfera = len(esfera)

### Empacotando os vértices

In [20]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices["position"] = vertices_list

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar um slot (buffer).

In [21]:
# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
# Make this buffer the default one
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Abaixo, nós enviamos todo o conteúdo da variável vertices.

In [22]:
# Upload data
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

### Associando variáveis do programa GLSL (Vertex Shader) com nossos dados

In [23]:
# Bind the position attribute
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)

### Em seguida, solicitamos à GPU a localização da variável position

In [24]:
loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)

### A partir da localização anterior, nós indicamos à GPU onde está o conteúdo da variável position

In [25]:
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Vamos pegar a localização da variável color para definir a cor de cada forma

In [26]:
loc_color = glGetUniformLocation(program, "color")

### Matrizes de transformação

In [27]:
def multiplica_matriz(a, b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a, m_b)
    c = m_c.reshape(1,16)
    return c


def matriz_rotacao_x(angulo):
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([1.0,   0.0,    0.0, 0.0,
                     0.0, cos_d, -sin_d, 0.0,
                     0.0, sin_d,  cos_d, 0.0,
                     0.0,   0.0,    0.0, 1.0], np.float32)


def matriz_rotacao_y(angulo):
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([ cos_d, 0.0, sin_d, 0.0,
                      0.0,   1.0,  0.0, 0.0,
                     -sin_d, 0.0, cos_d, 0.0,
                      0.0,   0.0,  0.0, 1.0], np.float32)


def matriz_rotacao_z(angulo):
    cos_d = math.cos(angulo)
    sin_d = math.sin(angulo)
    return np.array([cos_d, -sin_d, 0.0, 0.0,
                     sin_d,  cos_d, 0.0, 0.0,
                     0.0,    0.0,   1.0, 0.0,
                     0.0,    0.0,   0.0, 1.0], np.float32)


def matriz_translacao(t_x, t_y, t_z):
    return np.array([1.0, 0.0, 0.0, t_x,
                     0.0, 1.0, 0.0, t_y,
                     0.0, 0.0, 1.0, t_z,
                     0.0, 0.0, 0.0, 1.0], np.float32)


def matriz_escala(s_x, s_y, s_z):
    return np.array([s_x, 0.0, 0.0, 0.0,
                     0.0, s_y, 0.0, 0.0,
                     0.0, 0.0, s_z, 0.0,
                     0.0, 0.0, 0.0, 1.0], np.float32)


### Variáveis das transformações e funções para desenhar cada forma

In [28]:
angulo_cubo = 0.0
angulo_piramide = 0.0


def desenha_cubo():
    global angulo_cubo

    mat_rotation_y = matriz_rotacao_y(angulo_cubo)
    mat_rotation_x = matriz_rotacao_x(angulo_cubo * 0.6)
    mat_scale = matriz_escala(0.18, 0.18, 0.18)
    mat_translation = matriz_translacao(-0.70, 0.45, 0.0)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_rotation_x)
    mat_transform = multiplica_matriz(mat_scale, mat_transform)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.85, 0.20, 0.20, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cubo, quantos_vertices_cubo)


def desenha_piramide():
    global angulo_piramide

    mat_rotation_x = matriz_rotacao_x(angulo_piramide)
    mat_rotation_z = matriz_rotacao_z(angulo_piramide * 0.5)
    mat_scale = matriz_escala(0.20, 0.20, 0.20)
    mat_translation = matriz_translacao(-0.20, 0.45, 0.0)

    mat_transform = multiplica_matriz(mat_rotation_x, mat_rotation_z)
    mat_transform = multiplica_matriz(mat_scale, mat_transform)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.20, 0.70, 0.30, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_piramide, quantos_vertices_piramide)


def desenha_prisma():

    mat_rotation_y = matriz_rotacao_y(math.radians(30.0))
    mat_scale = matriz_escala(0.20, 0.20, 0.20)
    mat_translation = matriz_translacao(0.30, 0.45, 0.0)

    mat_transform = multiplica_matriz(mat_scale, mat_rotation_y)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.20, 0.45, 0.85, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_prisma, quantos_vertices_prisma)


def desenha_cilindro():
    global t_x_cilindro, t_y_cilindro

    mat_scale = matriz_escala(0.16, 0.16, 0.16)
    mat_translation = matriz_translacao(t_x_cilindro, t_y_cilindro, 0.0)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.80, 0.55, 0.15, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cilindro, quantos_vertices_cilindro)


def desenha_cone():
    global angulo_cone

    mat_rotation_y = matriz_rotacao_y(angulo_cone)
    mat_rotation_z = matriz_rotacao_z(math.radians(-35.0))
    mat_scale = matriz_escala(0.18, 0.18, 0.18)
    mat_translation = matriz_translacao(-0.45, -0.35, 0.0)

    mat_transform = multiplica_matriz(mat_rotation_y, mat_rotation_z)
    mat_transform = multiplica_matriz(mat_scale, mat_transform)
    mat_transform = multiplica_matriz(mat_translation, mat_transform)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.65, 0.25, 0.80, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_cone, quantos_vertices_cone)


def desenha_esfera():
    global escala_esfera

    mat_scale = matriz_escala(escala_esfera, escala_esfera, escala_esfera)
    mat_translation = matriz_translacao(0.35, -0.35, 0.0)

    mat_transform = multiplica_matriz(mat_translation, mat_scale)

    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_transform)

    glUniform4f(loc_color, 0.20, 0.75, 0.75, 1.0)
    glDrawArrays(GL_TRIANGLES, inicio_esfera, quantos_vertices_esfera)


### Nesse momento, nós exibimos a janela

In [29]:
glfw.show_window(window)

### Loop principal da janela

In [30]:
glEnable(GL_DEPTH_TEST)

while not glfw.window_should_close(window):

    glfw.poll_events()

    angulo_cubo += 0.01
    angulo_piramide -= 0.02

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    if polygonal_mode == True:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    desenha_cubo()
    desenha_piramide()
    desenha_prisma()
    desenha_cilindro()
    desenha_cone()
    desenha_esfera()

    glfw.swap_buffers(window)

glfw.terminate()


### Observação

Cada forma possui sua própria matriz de transformação.

Teclas:
- `P`: alterna entre preenchido e malha
- `← ↑ ↓ →`: translada o cilindro
- `A` e `Z`: altera a escala da esfera
- `Q` e `E`: rotaciona o cone
